# Figures A1 and A4

In [0]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrow
#from matplotlib_scalebar.scalebar import ScaleBar
import contextily as cx
import matplotlib.patheffects as pe

#Figure A4

In [0]:
DATA_DIR = "/Workspace/Repos/mbartlett@thewaterinstitute.org/extended-JPM/reports/data"   # local file API path for /dbfs
HWM_PATH    = f"{DATA_DIR}/HWMs.geojson"
GAGE_PATH   = f"{DATA_DIR}/usgs_gages.geojson"
RIVER_PATH  = f"{DATA_DIR}/Major_Rivers.geojson"
HMS_GAGES_PATH = f"{DATA_DIR}/HMS_USGS_gages.geojson"
HMS_BASINS_PATH = f"{DATA_DIR}/HMS_basins.geojson"
HMS_RIVERS_PATH = f"{DATA_DIR}/HMS_rivers.geojson"
DOMAIN_PATH = f"{DATA_DIR}/HEC-RAS-domain.geojson"


OUT_PDF = f"{DATA_DIR}/hwm_usgs_domain_figure.pdf"
 
# Map projection used for plotting + basemap tiles (Web Mercator).
PLOT_CRS = "EPSG:3857"
 
# IMPORTANT — HEC-RAS domain CRS.
# The domain GeoJSON has NO embedded CRS and its coordinates do not match any standard
# EPSG zone tested (Louisiana State Plane, UTM 15N, CONUS Albers were all checked and rejected).
# It is most likely a custom RAS Mapper projection. Set the correct CRS here if known.
# The auto-detect step below will try candidates and warn if the domain does not overlap the data.
HECRAS_DOMAIN_CRS = "EPSG:3452"   # Louisiana South State Plane (NAD83, ftUS) — best guess; override if wrong
 
# Storm color map for HWMs (keyed on the 'Name' property)
STORM_COLORS = {
    "Katrina": "#d62728",   # red
    "Isaac":   "#1f77b4",   # blue
    "Gustav":  "#2ca02c",   # green
}
GAGE_COLOR  = "#ff7f0e"     # orange
RIVER_COLOR = "#3b78c2"
DOMAIN_EDGE = "#222222"

## 3. Load layers

In [0]:
hwm    = gpd.read_file(HWM_PATH)
gages  = gpd.read_file(GAGE_PATH)
rivers = gpd.read_file(RIVER_PATH)
domain = gpd.read_file(DOMAIN_PATH)
hms_gages = gpd.read_file(HMS_GAGES_PATH)
hms_basins = gpd.read_file(HMS_BASINS_PATH)
hms_rivers = gpd.read_file(HMS_RIVERS_PATH)

In [0]:
hwm    = gpd.read_file(HWM_PATH)
gages  = gpd.read_file(GAGE_PATH)
rivers = gpd.read_file(RIVER_PATH)
domain = gpd.read_file(DOMAIN_PATH)
hms_gages = gpd.read_file(HMS_GAGES_PATH)
hms_basins = gpd.read_file(HMS_BASINS_PATH)
hms_rivers = gpd.read_file(HMS_RIVERS_PATH)
 
# Point layers and rivers are in WGS84 / NAD83 (lon-lat). Assign if missing.
for gdf in (hwm, gages, rivers, hms_gages, hms_basins, hms_rivers):
    if gdf.crs is None:
        gdf.set_crs("EPSG:4326", inplace=True)
 
print("HWMs:   ", len(hwm),   "features | storms:", sorted(hwm["Name"].unique()))
print("Gages:  ", len(gages), "features")
print("Rivers: ", len(rivers),"features | crs:", rivers.crs)
print("Domain: ", len(domain),"features | crs:", domain.crs)

print("HMS Rivers: ", len(hms_rivers),"features | crs:", hms_rivers.crs)
print("HMS Basins: ", len(hms_basins),"features | crs:", hms_basins.crs)
print("HMS USGS Gages: ", len(hms_gages),"features | crs:", hms_gages.crs)

In [0]:
wkt_string = 'PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["false_easting",0.0],PARAMETER["false_northing",0.0],PARAMETER["central_meridian",-96.0],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["latitude_of_origin",23.0],UNIT["Foot_US",0.3048006096012192]]'

domain = domain.set_crs(wkt_string, allow_override=True)

fig, ax = plt.subplots(figsize=(12, 10))

# Plot domain boundary
domain_plot = domain.to_crs(PLOT_CRS)
domain_plot.plot(ax=ax, edgecolor=DOMAIN_EDGE, facecolor='none', linewidth=2, label='HEC-RAS Domain')

# Adjust y-axis to extend 1 inch above the domain
bounds = domain_plot.total_bounds  # [minx, miny, maxx, maxy]
minx, miny, maxx, maxy = bounds
dpi = fig.dpi
fig_height_inch = fig.get_figheight()
ax_height_pix = fig_height_inch * dpi
y_range = maxy - miny
pix_per_unit = ax_height_pix / y_range
inch_above = dpi  # 1 inch in pixels
y_extra = inch_above / pix_per_unit
ax.set_ylim(miny, maxy + y_extra)

# Plot rivers UNDER the HWM and USGS markers
rivers.to_crs(PLOT_CRS).plot(ax=ax, color=RIVER_COLOR, linewidth=1.5, label='Major Rivers', zorder=1)

# Plot gages with white outline
gages_plot = gages.to_crs(PLOT_CRS)
gages_plot.plot(ax=ax, color=GAGE_COLOR, markersize=60, marker='^', label='USGS Gages', edgecolor='white', linewidth=1.5, zorder=3)

# Plot HWMs by storm color with white outline
for storm, color in STORM_COLORS.items():
    hwm_storm = hwm[hwm["Name"] == storm].to_crs(PLOT_CRS)
    hwm_storm.plot(ax=ax, color=color, markersize=50, marker='o', label=f'HWM: {storm}', edgecolor='white', linewidth=1.5, zorder=4)

# Add basemap
cx.add_basemap(ax, crs=PLOT_CRS)

###LABELS###

import numpy as np
import matplotlib.patheffects as pe
from shapely.ops import unary_union

# --- Label the 4 major rivers where each reach is most separated from the others ---
rivers_plot = rivers.to_crs(PLOT_CRS)

def _longest(geom):
    return max(geom.geoms, key=lambda g: g.length) if geom.geom_type == "MultiLineString" else geom

def _tangent_at(line, frac, eps=0.75):
    p1 = line.interpolate(max(0.0, frac - eps), normalized=True)
    p2 = line.interpolate(min(1.0, frac + eps), normalized=True)
    ang = np.degrees(np.arctan2(p2.y - p1.y, p2.x - p1.x))
    if ang > 90:  ang -= 180
    if ang < -90: ang += 180
    return ang

river_lines = {
    row["gnis_name"]: _longest(row.geometry)
    for _, row in rivers_plot.iterrows()
    if row.get("gnis_name")
}

LABEL_OFFSET = 1200  # meters perpendicular to the channel; flip sign to label other side

for name, line in river_lines.items():
    others = unary_union([l for n, l in river_lines.items() if n != name])

    # pick the point along this reach with the most clearance from the other rivers
    best_frac, best_clear = 0.5, -1.0
    for frac in np.linspace(0.1, 0.9, 33):
        clear = line.interpolate(frac, normalized=True).distance(others)
        if clear > best_clear:
            best_clear, best_frac = clear, frac

    pt = line.interpolate(best_frac, normalized=True)
    ang = _tangent_at(line, best_frac)

    perp = np.radians(ang + 90)
    ox, oy = np.cos(perp) * LABEL_OFFSET, np.sin(perp) * LABEL_OFFSET

    # Move 'Tickfaw River' label to the left by subtracting from ox
    if name == "Tickfaw River":
        ox -= 4000  # adjust this value as needed for desired shift
        oy += 1000

    ax.text(
        pt.x + ox, pt.y + oy, name,
        fontsize=10, fontstyle="italic", color="#1a3c6e",
        ha="center", va="center",
        rotation=ang, rotation_mode="anchor",
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
        zorder=10,
    )

# --- Label the two lakes at fixed geographic points ---
lake_labels = {
    "Lake Maurepas":     (-90.50, 30.27),
    "Lake Pontchartrain": (-90.20, 30.20),
}
lake_pts = (
    gpd.GeoDataFrame(
        {"name": list(lake_labels.keys())},
        geometry=gpd.points_from_xy(
            [c[0] for c in lake_labels.values()],
            [c[1] for c in lake_labels.values()],
        ),
        crs="EPSG:4326",
    ).to_crs(PLOT_CRS)
)
for _, row in lake_pts.iterrows():
    ax.text(
        row.geometry.x, row.geometry.y, row["name"],
        fontsize=11, fontweight="bold", fontstyle="italic", color="#15616d",
        ha="center", va="center",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        zorder=10,
    )

##############

# Legend
handles = [
    mlines.Line2D([], [], color=DOMAIN_EDGE, linewidth=2, label='HEC-RAS Domain'),
    mlines.Line2D([], [], color=RIVER_COLOR, linewidth=1.5, label='Major Rivers'),
    mlines.Line2D([], [], color=GAGE_COLOR, marker='^', linestyle='', markersize=10, markeredgewidth=1.5, markeredgecolor='white', label='USGS Gages'),
]
for storm, color in STORM_COLORS.items():
    handles.append(mlines.Line2D([], [], color=color, marker='o', linestyle='', markersize=10, markeredgewidth=1.5, markeredgecolor='white', label=f'HWM: {storm}'))
ax.legend(handles=handles, loc='upper right')

#ax.set_title("HEC-RAS Domain, Major Rivers, USGS Gages, and HWMs")
ax.axis('off')

plt.tight_layout()
plt.savefig(OUT_PDF, dpi=300)
plt.show()

In [0]:
OUT_PDF

# Figure A4 Analysis

## 4. Resolve the HEC-RAS domain CRS
The domain carries no CRS. We assign the configured guess, then sanity-check by reprojecting to lon/lat and confirming it overlaps the HWM/gage extent. If it does not, a list of candidates is tried automatically and a warning is printed so you can correct `HECRAS_DOMAIN_CRS` above.

In [0]:
hms_gages = hms_gages[hms_gages["Station_Name"].isin([
    "Amite River near Denham Springs, LA",
    "Tickfaw River at Holden, LA",
    "Natalbany River at Baptist, LA",
    "Tangipahoa River at Robert, LA"
])]

In [0]:
import geopandas as gpd
import contextily as cx
import matplotlib.lines as mlines

crs_102039_ft = '+proj=aea +lat_1=29.5 +lat_2=45.5 +lat_0=23 +lon_0=-96 +x_0=0 +y_0=0 +datum=NAD83 +units=us-ft'

wkt_string = 'PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["false_easting",0.0],PARAMETER["false_northing",0.0],PARAMETER["central_meridian",-96.0],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["latitude_of_origin",23.0],UNIT["Foot_US",0.3048006096012192]]'

domain = domain.set_crs(wkt_string, allow_override=True)

hms_basins = gpd.read_file(HMS_BASINS_PATH).set_crs(crs_102039_ft, allow_override=True)
hms_rivers = gpd.read_file(HMS_RIVERS_PATH).set_crs(crs_102039_ft, allow_override=True)

# Reproject to Web Mercator for plotting with basemap
hms_basins = hms_basins.to_crs(epsg=3857)
hms_rivers = hms_rivers.to_crs(epsg=3857)
hms_gages= hms_gages.to_crs(epsg=3857)
domain_plot = domain.to_crs(PLOT_CRS)

hms_gages = hms_gages[hms_gages["SITENAME"].isin([
    "Amite River near Denham Springs, LA",
    "Tickfaw River at Holden, LA",
    "Natalbany River at Baptist, LA",
    "Tangipahoa River at Robert, LA"
])]

hms_basins = hms_basins[~hms_basins["Code"].astype(str).str.contains("11")]

# River color mapping
river_colors = {
    "Upper Amite": "blue",
    "Lower Amite": "black",
    "Tickfaw": "purple",
    "Natalbany": "orange",
    "Tangipahoa": "green"
}

fig, ax = plt.subplots(figsize=(12, 18))
ax.axis('off')
domain_plot.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2, label='HEC-RAS Domain')

hms_basins.plot(ax=ax, facecolor='none', edgecolor='black', alpha=0.5)

# Plot each river by color
for name, color in river_colors.items():
    river_seg = hms_rivers[hms_rivers["GNIS_Name"] == name]
    if not river_seg.empty:
        river_seg.plot(ax=ax, color=color, linewidth=0.8, label=name)

hms_gages.plot(ax=ax, color='blue', linewidth=0.8)
cx.add_basemap(ax, crs="EPSG:3857")

# Legend handles
handles = [
    mlines.Line2D([], [], color='red', linewidth=2, label='HEC-RAS Domain'),
    mlines.Line2D([], [], color='none', linewidth=0, label='Major drainages', alpha=0)
]
for name, color in river_colors.items():
    handles.append(mlines.Line2D([], [], color=color, linewidth=2, label=name))
ax.legend(handles=handles, loc='upper right')

plt.show()

# Figure A4

In [0]:
crs_102039_ft = '+proj=aea +lat_1=29.5 +lat_2=45.5 +lat_0=23 +lon_0=-96 +x_0=0 +y_0=0 +datum=NAD83 +units=us-ft'

wkt_string = 'PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["false_easting",0.0],PARAMETER["false_northing",0.0],PARAMETER["central_meridian",-96.0],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["latitude_of_origin",23.0],UNIT["Foot_US",0.3048006096012192]]'

domain = domain.set_crs(wkt_string, allow_override=True)

hms_basins = gpd.read_file(HMS_BASINS_PATH).set_crs(crs_102039_ft, allow_override=True)
hms_rivers = gpd.read_file(HMS_RIVERS_PATH).set_crs(crs_102039_ft, allow_override=True)

# Reproject to Web Mercator for plotting with basemap
hms_basins = hms_basins.to_crs(epsg=3857)
hms_rivers = hms_rivers.to_crs(epsg=3857)
hms_gages= hms_gages.to_crs(epsg=3857)
domain_plot = domain.to_crs(PLOT_CRS)

hms_gages = hms_gages[hms_gages["SITENAME"].isin([
    "Amite River near Denham Springs, LA",
    "Tickfaw River at Holden, LA",
    "Natalbany River at Baptist, LA",
    "Tangipahoa River at Robert, LA"
])]

hms_basins2 = hms_basins[~hms_basins["Code"].astype(str).str.contains("11")]

In [0]:
# Shorten gage labels for map legibility (split on ", LA" or at comma)
def shorten_gage(name):
    """'Amite River near Denham Springs, LA' -> 'Amite R.\nnr. Denham Springs'"""
    replacements = {
        "Amite River near Denham Springs, LA":  "Amite R. nr.\nDenham Springs",
        "Tickfaw River at Holden, LA":           "Tickfaw R.\nat Holden",
        "Natalbany River at Baptist, LA":        "Natalbany R.\nat Baptist",
        "Tangipahoa River at Robert, LA":        "Tangipahoa R.\nat Robert",
    }
    return replacements.get(name, name)

# ── PATH CONFIGURATION ────────────────────────────────────────────────────────
OUTPUT_PATH     = "HMS_study_area.pdf"

# ── RIVER COLOR MAP ───────────────────────────────────────────────────────────
RIVER_COLORS = {
    "Upper Amite":  "#1565C0",
    "Lower Amite":  "#1A237E",
    "Tickfaw":      "#7B1FA2",
    "Natalbany":    "#E65100",
    "Tangipahoa":   "#2E7D32",
}

hms_gages["label"] = hms_gages["SITENAME"].apply(shorten_gage)

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE
# ─────────────────────────────────────────────────────────────────────────────
FIG_W, FIG_H = 6.5, 8.25   # inches: full text-block width, 3/4 of 11-in page
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), facecolor="white")

# ── Extent from basins + small padding ───────────────────────────────────────
b = hms_basins.total_bounds          # [xmin, ymin, xmax, ymax] in Web Mercator
pw = (b[2] - b[0]) * 0.05
ph = (b[3] - b[1]) * 0.05
ax.set_xlim(b[0] - pw, b[2] + pw)
ax.set_ylim(b[1] - ph, b[3] + ph)

# ── Topographic basemap (Esri World Topo) ────────────────────────────────────
cx.add_basemap(
    ax,
    crs=PLOT_CRS,
    source=cx.providers.Esri.WorldTopoMap,
    zoom=11,
    attribution_size=5,
)

# ── HEC-RAS domain boundary ──────────────────────────────────────────────────
domain.to_crs(PLOT_CRS).plot(ax=ax, facecolor="none", edgecolor="#C62828",
            linewidth=1.8, linestyle="--", zorder=3)

# ── Subbasin polygons ─────────────────────────────────────────────────────────
hms_basins2.plot(ax=ax, facecolor="none", edgecolor="#455A64",
                linewidth=1.8, alpha=0.8, zorder=4)

# ── Rivers by drainage ────────────────────────────────────────────────────────
for name, color in RIVER_COLORS.items():
    seg = hms_rivers[hms_rivers["GNIS_Name"] == name]
    if not seg.empty:
        seg.plot(ax=ax, color=color, linewidth=1.8, zorder=5)

# ── Basin Code labels (centered in each subbasin) ────────────────────────────
for _, row in hms_basins2.iterrows():
    cgx = row.geometry.centroid.x
    cgy = row.geometry.centroid.y
    ax.text(cgx, cgy, row["Code"],
            fontsize=7, fontweight="bold", ha="center", va="center",
            color="#1A237E", zorder=7,
            path_effects=[pe.withStroke(linewidth=2.0, foreground="white")])

# ── USGS gage markers ─────────────────────────────────────────────────────────
ax.scatter(hms_gages.geometry.x, hms_gages.geometry.y,
           marker="^", s=80, color="#B71C1C",
           edgecolors="white", linewidths=0.7, zorder=8)

# ── Gage labels (with white halo for legibility over topo) ───────────────────
for _, row in hms_gages.iterrows():
    # Move 'Tickfaw River at Holden, LA' label to the left by shifting x offset
    if row["SITENAME"] == "Tickfaw River at Holden, LA":
        offset = (-1.9, 3)  # left and slightly up
        ha = "right"
    else:
        offset = (6, 6)
        ha = "left"
    ax.annotate(
        row["label"],
        xy=(row.geometry.x, row.geometry.y),
        xytext=offset, textcoords="offset points",
        fontsize=7.0, color="#7B0000", fontweight="semibold",
        ha=ha, va="bottom", linespacing=1.3,
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
        zorder=9,
    )

# ── Lake labels ───────────────────────────────────────────────────────────────
lake_labels = {
    "Lake \n Maurepas":     (-90.50, 30.27),
    "Lake Ponchartrain": (-90.20, 30.20),
}
lake_pts = (
    gpd.GeoDataFrame(
        {"name": list(lake_labels.keys())},
        geometry=gpd.points_from_xy(
            [c[0] for c in lake_labels.values()],
            [c[1] for c in lake_labels.values()],
        ),
        crs="EPSG:4326",
    ).to_crs(PLOT_CRS)
)
for _, row in lake_pts.iterrows():
    ax.text(
        row.geometry.x, row.geometry.y, row["name"],
        fontsize=9, fontweight="bold", fontstyle="italic", color="#15616d",
        ha="center", va="center",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        zorder=10,
    )

# ── Scale bar ─────────────────────────────────────────────────────────────────
# 20 km in Web Mercator meters (true at this latitude ~30.5°N)
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
bar_m  = 20_000
bx0    = xmin + (xmax - xmin) * 0.05
by0    = ymin + (ymax - ymin) * 0.04
bh     = (ymax - ymin) * 0.010
# Two alternating 10-km segments
for i, col in enumerate(["black", "white"]):
    ax.barh(by0, bar_m / 2, left=bx0 + i * bar_m / 2,
            height=bh, color=col, edgecolor="black",
            linewidth=0.6, zorder=10)
for label, xpos in [("0", bx0), ("10", bx0 + bar_m/2), ("20 km", bx0 + bar_m)]:
    ax.text(xpos, by0 + bh * 2.0, label,
            ha="center", va="bottom", fontsize=6.5, zorder=10)

# ── North arrow ───────────────────────────────────────────────────────────────
arr_x = xmax - (xmax - xmin) * 0.07
arr_y = ymin + (ymax - ymin) * 0.06
arr_h = (ymax - ymin) * 0.055
ax.annotate("", xy=(arr_x, arr_y + arr_h), xytext=(arr_x, arr_y),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.8), zorder=10)
ax.text(arr_x, arr_y + arr_h * 1.05, "N",
        ha="center", va="bottom", fontsize=10, fontweight="bold", zorder=10)

# ── Legend ────────────────────────────────────────────────────────────────────
handles = [
    mlines.Line2D([], [], color="#C62828", linewidth=1.8,
                  linestyle="--", label="HEC-RAS Domain"),
    mpatches.Patch(facecolor="none", edgecolor="#455A64",
                   linewidth=0.7, label="HMS Subbasin"),
]
for name, color in RIVER_COLORS.items():
    handles.append(mlines.Line2D([], [], color=color, linewidth=2.2, label=f"{name} River"))
handles.append(
    mlines.Line2D([], [], color="#B71C1C", marker="^", markersize=8,
                  linestyle="none", markeredgecolor="white",
                  markeredgewidth=0.7, label="USGS Stream Gage")
)

ax.legend(
    handles=handles,
    loc="upper right",
    fontsize=7.5,
    title="Legend", title_fontsize=8,
    framealpha=0.92, edgecolor="#AAAAAA",
    handlelength=2.2, handletextpad=0.7,
    borderpad=0.9, labelspacing=0.5,
)

# ── Axes formatting ───────────────────────────────────────────────────────────
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_linewidth(0.6)
    spine.set_edgecolor("#888888")

plt.tight_layout(pad=0.3)
plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Saved to {OUTPUT_PATH}")